In [1]:
import pandas as pd
import numpy as np

In [2]:
#import data from /Users/jaytlinaskew/GitRepository/TimeSeries-Analysis/data/processed/ProcessedObservedData.csv
data = pd.read_csv(r'C:\Users\jaskew\Documents\project_repository\data\processed\ProcessedObservedData.csv')
#filter by opDiv CDC
data = data[data['OpDiv'] == 'CDC']
data.drop(columns=['curr_date', 'OpDiv'], inplace=True)
# Rename the index to 'date' since 'obs_date' is now the index
data.rename(columns={'obs_date': 'date'}, inplace=True)
data['date'] = pd.to_datetime(data['date'])
data.reset_index(drop=True, inplace=True)

data.tail(10)

,indicator,API_UserName,date,observations
2578,www.shorturl.at/,00818860012482918321,2025-04-23,21
2579,162.142.125.255,00818860012482918321,2025-04-24,1
2580,162.142.125.247,00818860012482918321,2025-04-24,3
2581,146.71.50.198,00818860012482918321,2025-04-24,1
2582,68.67.179.164,00818860012482918321,2025-04-24,1728
2583,104.21.48.1,00818860012482918321,2025-04-24,210
2584,34.160.111.145,00818860012482918321,2025-04-24,2
2585,84.239.31.15,00818860012482918321,2025-04-24,2
2586,64.64.112.145,00818860012482918321,2025-04-24,1
2587,46.246.8.73,00818860012482918321,2025-04-24,1


In [3]:
data['date'] = pd.to_datetime(data['date'])

# Define ranges for combinations
all_users = data['API_UserName'].unique()  # Unique API_UserName
all_indicators = data['indicator'].unique()  # Unique indicators
all_dates = pd.date_range(start='2025-01-01', end=pd.Timestamp.now().normalize(), freq='D')

# Create all combinations
all_combinations = pd.MultiIndex.from_product(
    [all_users, all_dates, all_indicators],
    names=['API_UserName', 'date', 'indicator']
).to_frame(index=False)

# Merge with the existing data
merged = all_combinations.merge(data, how='left', on=['API_UserName', 'date', 'indicator'])

# Fill missing values in 'observations' with 0 (not seen)
merged['observations'] = merged['observations'].fillna(0).astype(int)

# Display the first few rows of the merged dataset

# Convert the 'date' column to datetime format
merged['date'] = pd.to_datetime(merged['date'])

# Extract day of the week (0=Monday, 6=Sunday)
merged['dayofweek'] = merged['date'].dt.dayofweek

# Determine if the day is a weekend (Saturday=5, Sunday=6)
merged['is_weekend'] = merged['dayofweek'].isin([5, 6])

# Extract additional features if needed
merged['day'] = merged['date'].dt.day
merged['month'] = merged['date'].dt.month

merged['seen'] = (merged['observations'] > 0).astype(int)
merged.head(10)
merged.to_csv(r'C:\Users\jaskew\Documents\project_repository\notebooks\observationEventForecasting\DataPreprocessing\FullIndicatorMatrix.csv', index=False)